# H2 vs Kr Proton-Impact Ionisation Cross-Section Comparison

Loads tabulated cross sections, plots sigma(E), and highlights the 30 keV point.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'beam energy [keV]':   30.0,
    'cross-section source':'warpx-data/MCC_cross_sections/',
    'gases':               'H2, Kr',
    'energy range [keV]':  '1 - 1000',
}
print_simulation_config(
    notebook_title='H2 vs Kr Cross-Section Comparison',
    defaults=_DEFAULTS, overrides={},
)


## 1. Plot sigma(E)


In [ ]:
from plasma_column.gas import load_cross_section_table, get_h2_cross_section, get_kr_cross_section
from plasma_column.plotting import setup_publication_style, save_figure
setup_publication_style()

BEAM_KEV = 30.0
XS_ROOT  = WARPX_DATA_DIR / 'MCC_cross_sections'
GAS_META = {
    'H2': {'path': XS_ROOT / 'H2' / 'proton_impact_ionization.dat',
           'color': 'tab:blue',   'label': r'H$_2$'},
    'Kr': {'path': XS_ROOT / 'Kr' / 'proton_impact_ionization.dat',
           'color': 'tab:orange', 'label': 'Kr'},
}

fig, ax = plt.subplots(figsize=(9, 5))
for gas, meta in GAS_META.items():
    if not meta['path'].exists():
        print(f'{gas}: not found — {meta["path"]}')
        continue
    df_xs = load_cross_section_table(meta['path'])
    ax.loglog(df_xs.iloc[:,0] * 1e-3, df_xs.iloc[:,1],
              color=meta['color'], lw=2, label=meta['label'])

ax.axvline(BEAM_KEV, color='gray', lw=1.2, ls='--',
           label=f'{BEAM_KEV:.0f} keV operating point')
ax.set_xlabel('Proton kinetic energy [keV]', fontsize=12)
ax.set_ylabel(r'Cross section [m$^2$]', fontsize=12)
ax.set_title(r'Proton-impact ionisation: H$_2$ vs Kr', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, ls='--', alpha=0.5, which='both')
p, _ = save_figure(fig, PLOTS_DIR / 'h2_kr_cross_sections')
plt.show()
print('Saved:', p.name)


## 2. Operating-point read-out


In [ ]:
s_h2 = get_h2_cross_section(BEAM_KEV)
s_kr = get_kr_cross_section(BEAM_KEV)
print(f'sigma(H2, {BEAM_KEV} keV) = {s_h2:.4e} m2')
print(f'sigma(Kr, {BEAM_KEV} keV) = {s_kr:.4e} m2')
print(f'Ratio sigma_Kr / sigma_H2  = {s_kr/s_h2:.2f}')


## 3. Ionization time-constant τ vs gas pressure

Characteristic time for the plasma column to build up to steady-state neutralization.
Formula: τ = 1 / (n_gas · σ · v_beam). Vertical dashed lines mark the baseline operating pressures.

In [ ]:
from plasma_column.neutralization import gas_density_m3, ionization_tau_s, proton_beta_gamma_speed
from plasma_column.plotting import setup_publication_style, save_figure
setup_publication_style()

beta, gamma, v_beam = proton_beta_gamma_speed(BEAM_KEV)
pressures = np.logspace(-6, -3, 300)  # Torr

sigma_h2 = get_h2_cross_section(BEAM_KEV)
sigma_kr = get_kr_cross_section(BEAM_KEV)

tau_h2_ns = [ionization_tau_s(gas_density_m3(p), sigma_h2, v_beam) * 1e9 for p in pressures]
tau_kr_ns = [ionization_tau_s(gas_density_m3(p), sigma_kr, v_beam) * 1e9 for p in pressures]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(pressures, tau_h2_ns, "tab:blue",  lw=2.5, label=r"H$ ($\sigma={:.2e}$ m$^2$)".format(sigma_h2))
ax.loglog(pressures, tau_kr_ns, "tab:orange", lw=2.5, label=r"Kr ($\sigma={:.2e}$ m$^2$)".format(sigma_kr))

# Mark operating pressures
tau_h2_op = ionization_tau_s(gas_density_m3(1e-5), sigma_h2, v_beam) * 1e9
tau_kr_op = ionization_tau_s(gas_density_m3(1e-6), sigma_kr, v_beam) * 1e9
ax.axvline(1e-5, ls="--", color="tab:blue",   lw=1.3, alpha=0.7, label=f"H2 op. ({1e-5:.0e} Torr, τ={tau_h2_op:.0f} ns)")
ax.axvline(1e-6, ls="--", color="tab:orange", lw=1.3, alpha=0.7, label=f"Kr op. ({1e-6:.0e} Torr, τ={tau_kr_op:.0f} ns)")

ax.set_xlabel("Gas Pressure [Torr]", fontsize=12)
ax.set_ylabel(r"Ionization Time Constant $	au$ [ns]", fontsize=12)
ax.set_title("Ionization Time Constant vs Gas Pressure (30 keV Protons)", fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "tau_vs_pressure.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "tau_vs_pressure.pdf", bbox_inches="tight")
plt.show()
print(f"H2: tau={tau_h2_op:.1f} ns at 1e-5 Torr")
print(f"Kr: tau={tau_kr_op:.1f} ns at 1e-6 Torr")

## 4. Neutralization build-up η(t) — H₂ pressure family

Analytic build-up curves η(t) = 1 − exp(−t/τ) for H₂ at multiple pressures.
Higher pressure → shorter τ → faster saturation.

In [ ]:
from plasma_column.neutralization import neutralization_fraction

pressures_scan = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4]  # Torr
t_ns = np.linspace(0, 2000, 500)
t_s  = t_ns * 1e-9

cmap_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(pressures_scan)))

fig, ax = plt.subplots(figsize=(9, 5))
for p_torr, col in zip(pressures_scan, cmap_colors):
    n_gas = gas_density_m3(p_torr)
    tau   = ionization_tau_s(n_gas, sigma_h2, v_beam)
    eta_t = neutralization_fraction(t_s, tau, eta_ss=1.0)
    ax.plot(t_ns, eta_t, color=col, lw=2,
            label=f"{p_torr:.0e} Torr  (τ={tau*1e9:.0f} ns)")

ax.axhline(0.5, ls=":",  color="black", lw=1.2, alpha=0.6, label="η = 0.5")
ax.axhline(0.9, ls="--", color="black", lw=1.2, alpha=0.6, label="η = 0.9")

# Colorbar proxy
import matplotlib.cm as cmx
import matplotlib.colors as mcolors
norm = mcolors.LogNorm(vmin=min(pressures_scan), vmax=max(pressures_scan))
sm   = plt.cm.ScalarMappable(cmap="plasma", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Gas Pressure [Torr]", fontsize=10)

ax.set_xlabel(r"Time $ [ns]", fontsize=12)
ax.set_ylabel(r"Neutralization Fraction $\eta(t)$", fontsize=12)
ax.set_title(r"Neutralization Build-up $\eta(t)$ — H$ at Various Pressures", fontsize=13)
ax.legend(fontsize=8, loc="center right")
ax.grid(True, ls="--", alpha=0.4)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eta_buildup_h2_pressures.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "eta_buildup_h2_pressures.pdf", bbox_inches="tight")
plt.show()
print("Saved eta_buildup_h2_pressures.{png,pdf}")

## 5. 2-D neutralization map: pressure × interaction length

Color: η_net at t=500 ns as a function of gas pressure and plasma column length.
Longer column → effectively longer exposure time for beam ions → faster neutralization.

In [ ]:
# At t=500 ns, eta = 1 - exp(-500ns / tau)
# Effective exposure time for a particle crossing length L at speed v: t_exp = L / v_beam
L_vals_m = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.50])  # column length [m]
p_vals   = np.logspace(-6, -3, 60)  # Torr

# Compute eta grid using H2
eta_grid = np.zeros((len(L_vals_m), len(p_vals)))
for i, L in enumerate(L_vals_m):
    t_exp_s = L / v_beam  # transit time through column
    for j, p in enumerate(p_vals):
        n_gas = gas_density_m3(p)
        tau   = ionization_tau_s(n_gas, sigma_h2, v_beam)
        eta_grid[i, j] = neutralization_fraction(t_exp_s, tau, eta_ss=1.0)

fig, ax = plt.subplots(figsize=(10, 5))
pcm = ax.pcolormesh(p_vals, L_vals_m * 100, eta_grid,
                    cmap="RdYlGn", vmin=0, vmax=1, shading="auto")
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r"Neutralization $\eta$ at beam transit", fontsize=10)

# Contours
cs = ax.contour(p_vals, L_vals_m * 100, eta_grid,
                levels=[0.3, 0.5, 0.7, 0.9], colors="white", linewidths=1.2, linestyles="--")
ax.clabel(cs, fmt="%.1f", fontsize=8, inline=True)

# Operating point: L=20 cm, p=1e-5 Torr (H2 baseline)
ax.plot(1e-5, 20.0, "w*", ms=14, zorder=5, label="H2 baseline (1e-5 Torr, 20 cm)")

ax.set_xscale("log")
ax.set_xlabel("Gas Pressure [Torr]", fontsize=12)
ax.set_ylabel("Plasma Column Length $ [cm]", fontsize=12)
ax.set_title(r"2-D Neutralization Map: Pressure × Column Length (H$, single transit)", fontsize=13)
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "neutralization_2d_pressure_length.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "neutralization_2d_pressure_length.pdf", bbox_inches="tight")
plt.show()
print("Saved neutralization_2d_pressure_length.{png,pdf}")